In [ ]:
%pip install elasticsearch==8.19.0
%restart_python

### Patch awards-v4 with the funding-flows fields (oxjob #1570)

Before the first nightly that serves them, PUT the mapping for the fields `nightly_grants` adds in the funding-flows release:
`amount_usd`, `amount_usd_basis`, `recipient_countries` (nested, display) + `recipient_countries_full` (object, filters),
`recipient_regions` + `recipient_regions_full`, `research_scope`. Same pattern as `PatchAwardsV4Mapping.ipynb` (oxjob #123.2):
the mapping goes in first so Elasticsearch does not guess it; the nightly's own sync then carries the values in `_source`.
No re-sync here: the release night restamps every award that gains a field, and the nightly upserts those documents itself.

**After this:** the elastic-api change (`awards/schemas.py`, `awards/fields.py`: `amount_usd` as a RangeField so
`sort=amount_usd.sum` works, `recipient_countries.country_code` + continent through the alias map, `research_scope.is_research`
as a BooleanField; `core/properties.py` version bump).

In [ ]:
import json
from pyspark.sql import functions as F
from elasticsearch import Elasticsearch, helpers
import logging

logging.basicConfig(level=logging.WARNING, format='[%(asctime)s]: %(message)s')
log = logging.getLogger(__name__)

ELASTIC_URL = dbutils.secrets.get(scope="elastic", key="elastic_url")

CONFIG = {
    "table_name": "openalex.awards.awards_api",
    "target_index": "awards-v4",
}

client = Elasticsearch(
    hosts=[ELASTIC_URL],
    max_retries=3,
    request_timeout=180,
)

assert client.indices.exists(index=CONFIG['target_index']), \
    f"{CONFIG['target_index']} not found; run BuildAwardsV4.ipynb first"
print(f"OK: {CONFIG['target_index']} exists")

In [ ]:
patch = {
    "properties": {
        "amount_usd":       {"type": "double"},
        "amount_usd_basis": {"type": "keyword"},
        "recipient_countries": {                      # nested: rich display, one object per country
            "type": "nested",
            "properties": {
                "country_code": {"type": "keyword", "fields": {"lower": {"type": "keyword", "normalizer": "lower"}}},
                "share":        {"type": "float"},
                "amount_usd":   {"type": "double"},
                "basis":        {"type": "keyword"},
            },
        },
        "recipient_countries_full": {                 # object sibling for term filters (institution_awarded_full pattern)
            "properties": {
                "country_code": {"type": "keyword", "fields": {"lower": {"type": "keyword", "normalizer": "lower"}}},
                "share":        {"type": "float"},
                "amount_usd":   {"type": "double"},
                "basis":        {"type": "keyword"},
            },
        },
        "recipient_regions": {
            "type": "nested",
            "properties": {
                "code":         {"type": "keyword"},
                "vocabulary":   {"type": "keyword"},
                "display_name": {"type": "text", "fields": {"keyword": {"type": "keyword"}}},
                "share":        {"type": "float"},
            },
        },
        "recipient_regions_full": {
            "properties": {
                "code":         {"type": "keyword"},
                "vocabulary":   {"type": "keyword"},
                "display_name": {"type": "text", "fields": {"keyword": {"type": "keyword"}}},
                "share":        {"type": "float"},
            },
        },
        "research_scope": {
            "properties": {
                "is_research": {"type": "boolean"},
                "basis":       {"type": "keyword"},
                "tier":        {"type": "integer"},
            },
        },
    }
}
print(json.dumps(patch, indent=2))
client.indices.put_mapping(index=CONFIG['target_index'], body=patch)
print("mapping patched")

In [ ]:
# Verify: the new fields are in the mapping; no document has them yet before the release night.
m = client.indices.get_mapping(index=CONFIG['target_index'])[CONFIG['target_index']]['mappings']['properties']
for f in ("amount_usd", "amount_usd_basis", "recipient_countries", "recipient_countries_full", "recipient_regions", "recipient_regions_full", "research_scope"):
    assert f in m, f
    print(f, m[f].get("type", "object"))
for f in ("amount_usd", "recipient_countries_full.country_code", "research_scope.is_research"):
    n = client.search(index=CONFIG['target_index'], body={"size": 0, "query": {"exists": {"field": f}}})['hits']['total']['value']
    print(f"documents with {f}: {n}")